In [1]:
# Instalare Unsloth optimizată pentru Colab
!pip install --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sv3zguiz/unsloth_e4c5de5e431446fa8661a95a24d311ef
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sv3zguiz/unsloth_e4c5de5e431446fa8661a95a24d311ef
  Resolved https://github.com/unslothai/unsloth.git to commit 48cf5ff213f653633a2884be7f50f8d2fe42ef3c
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 217.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 445.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 277.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 276.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 470.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 356.3 MB/s eta 0:00:00
 

In [2]:
import os
import torch
from unsloth import FastLanguageModel
from datetime import datetime
from datasets import load_dataset
from huggingface_hub import login
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

# --- 1. CONSTANTE PROIECT ---
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct"  # Varianta optimizată și instruită pentru chat/JSON
PROJECT_NAME = "cv-structuring-llm-v2"
HF_USER = "alecs-vlad24"                      # Contul tău de Hugging Face
DATASET_NAME = f"{HF_USER}/cv-resume-structuring-v2pro"

# Setăm numele rulării pentru salvare
RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M}"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# --- 2. HIPER-PARAMETRI - GENERALI ---
EPOCHS = 1  # Începem cu 1 epocă pentru test, putem crește ulterior
MAX_SEQUENCE_LENGTH = 4096  # CRUCIAL: CV-urile sunt lungi! Unsloth gestionează asta perfect.
BATCH_SIZE = 2              # Mic, pentru a încăpea CV-urile în memoria video
GRADIENT_ACCUMULATION_STEPS = 4  # 2 x 4 = 8 (Batch Size efectiv)

# --- 3. HIPER-PARAMETRI - QLoRA ---
QUANT_4_BIT = True
LORA_R = 16          # Standard optim pentru adaptare (16 sau 32)
LORA_ALPHA = 16
LORA_DROPOUT = 0     # Unsloth recomandă 0 pentru performanță maximă
# Unsloth detectează automat toate straturile necesare ("q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj")
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# --- 4. HIPER-PARAMETRI - ANTRENAMENT ---
LEARNING_RATE = 2e-4
WARMUP_RATIO = 0.05
LR_SCHEDULER_TYPE = 'linear'
WEIGHT_DECAY = 0.01
OPTIMIZER = "adamw_8bit"  # Optimizat de Unsloth pentru a economisi memorie

# Verificăm capabilitatea GPU-ului (T4 pe Colab nu suportă bf16 nativ, suportă fp16)
capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# --- 5. LOGGING ȘI TRACKING ---
SAVE_STEPS = 200
LOG_STEPS = 10
LOG_TO_WANDB = True  # Setăm pe False pentru a nu ne complica inițial cu conturi extra, dar poți pune True

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from google.colab import userdata

# IMPORTANT: Asigură-te că adaugi token-ul HF în tab-ul "Secrets" (cheia din stânga în Colab) sub numele 'HF_TOKEN'
hf_token = userdata.get('HF_TOKEN')
login(token=hf_token, add_to_git_credential=True)

In [4]:
wandb_api_key = userdata.get('WANDB_API_KEY')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: alecsvlad24 (alecsvlad24-n-a) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
dataset = load_dataset(DATASET_NAME)

# Preluăm seturile de date exact cum ai cerut (fără să mai tăiem din 'val', deoarece setul tău e deja mic și optim)
train = dataset['train']
# Hugging Face uneori redenumește partiția 'val' în 'validation' la upload. Verificăm ambele:
val = dataset['val'] if 'val' in dataset else dataset['validation']
test = dataset['test']

print(f"Seturi încărcate: Train={len(train)}, Val={len(val)}, Test={len(test)}")

README.md:   0%|          | 0.00/853 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.67MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.07MB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2798 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/350 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/350 [00:00<?, ? examples/s]

Seturi încărcate: Train=2798, Val=350, Test=350


In [6]:
if LOG_TO_WANDB:
  wandb.init(project=PROJECT_NAME, name=RUN_NAME)

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


In [7]:
# 1. Încărcăm modelul de bază în format 4-bit (QLoRA)
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQUENCE_LENGTH,
    dtype = None,               # Se adaptează automat (fp16 pentru T4)
    load_in_4bit = QUANT_4_BIT,
)

# 2. Configuram straturile LoRA pe care le vom antrena
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Foarte important pentru VRAM pe secvențe de 4096
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

print(f"Model și adaptor LoRA încărcate cu succes!")

==((====))==  Unsloth 2026.8.15: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.
Unsloth 2026.8.15 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Model și adaptor LoRA încărcate cu succes!


In [8]:
from unsloth.chat_templates import get_chat_template

# 1. Definim tokenizer-ul cu template-ul corect
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # Complet compatibil cu Llama-3.2-3B-Instruct
)

# 2. Funcția de conversie
def apply_chat_template_to_dataset(examples):
    texts = [tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False) for msg in examples["messages"]]
    return {"text": texts}

# 3. Aplicăm conversia
train = train.map(apply_chat_template_to_dataset, batched=True)
val = val.map(apply_chat_template_to_dataset, batched=True)

print("Formatarea ChatML aplicată cu succes!")

Map:   0%|          | 0/2798 [00:00<?, ? examples/s]

Map:   0%|          | 0/350 [00:00<?, ? examples/s]

Formatarea ChatML aplicată cu succes!


In [9]:
from trl import SFTTrainer, SFTConfig

# 1. Configurarea argumentelor folosind SFTConfig (aceiași parametri exact!)
sft_config = SFTConfig(
    dataset_text_field = "text",
    max_seq_length = MAX_SEQUENCE_LENGTH,
    dataset_num_proc = 2,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRADIENT_ACCUMULATION_STEPS,
    warmup_ratio = WARMUP_RATIO,
    num_train_epochs = EPOCHS,
    learning_rate = LEARNING_RATE,
    fp16 = not use_bf16,
    bf16 = use_bf16,
    logging_steps = LOG_STEPS,
    optim = OPTIMIZER,
    weight_decay = WEIGHT_DECAY,
    lr_scheduler_type = LR_SCHEDULER_TYPE,
    seed = 42,
    output_dir = PROJECT_RUN_NAME,
    report_to = "wandb" if LOG_TO_WANDB else "none",
    run_name = RUN_NAME,
    eval_strategy = "epoch",
    save_strategy = "epoch",
    load_best_model_at_end = True,
    save_safetensors = True,  # Previne eroarea de pickle la salvare
)

# 2. Crearea obiectului Trainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train,
    eval_dataset = val,
    args = sft_config,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: `save_safetensors` is not a valid SFTConfig argument for the installed TRL and will be IGNORED. Check the spelling, or your TRL version if this argument used to work.
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/2798 [00:00<?, ? examples/s]

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/350 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [10]:
print("Incepem antrenamentul! Asta va dura ceva timp...")

# 3. KICK-OFF: Rulăm antrenamentul efectiv!
trainer_stats = trainer.train()

# 4. Finalizăm sesiunea de wandb dacă e activată
if LOG_TO_WANDB:
    wandb.finish()

# 5. După ce s-a terminat, urcăm modelul tău antrenat pe Hugging Face
print(f"Antrenament finalizat. Urcăm modelul pe {HUB_MODEL_NAME}...")

Incepem antrenamentul! Asta va dura ceva timp...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,798 | Num Epochs = 1 | Total steps = 350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss
1,0.896847,1.000114


Unsloth: Restored added_tokens_decoder metadata in cv-structuring-llm-v2-2026-08-12_10.49/checkpoint-350/tokenizer_config.json.


eval/loss,▁
eval/runtime,▁
eval/samples_per_second,▁
eval/steps_per_second,▁
train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇█████
train/grad_norm,▄█▂▁▁▁▁▂▂▂▂▁▂▂▂▂▂▂▂▂▁▂▂▂▂▂▂▂▂▂▂▂▂▁▂
train/learning_rate,▄███▇▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▃▂▂▂▂▂▁▁▁
train/loss,█▆▃▂▂▂▂▂▂▂▁▁▁▁▁▂▁▁▂▂▁▁▂▂▂▂▁▂▁▂▁▂▁▂▁
eval/loss,1.00011
eval/runtime,902.2353


Antrenament finalizat. Urcăm modelul pe alecs-vlad24/cv-structuring-llm-v2-2026-08-12_10.49...


In [11]:
model.push_to_hub(HUB_MODEL_NAME, token = hf_token)
tokenizer.push_to_hub(HUB_MODEL_NAME, token = hf_token)
print("Succes!")

README.md:   0%|          | 0.00/586 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          | 45.8kB / 97.3MB            

Saved model to https://huggingface.co/alecs-vlad24/cv-structuring-llm-v2-2026-08-12_10.49


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmp48uozuqo/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mp48uozuqo/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

Succes!
